# 04 — Interest Rate Swaps

This notebook covers:
- **Vanilla IRS** (payer/receiver): NPV, fair rate, cashflow extraction
- **DV01** via automatic differentiation
- **vmap**: batch pricing across 1,000 rate scenarios
- **Cashflow inspection**: fixed and floating legs

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare, compare_table, timed_ms, plot_speedup
jax = setup_backend(BACKEND)
import jax.numpy as jnp
QL = load_quantlib()

---
## 1. Vanilla Interest Rate Swap

A **payer swap** pays fixed, receives floating. Its NPV:

$$
\text{NPV}_{\text{payer}} = \sum_{j} N \cdot \tau_j^{\text{float}} \cdot (L_j + s) \cdot P(0, t_j^{\text{float}})
- \sum_{i} N \cdot \tau_i^{\text{fix}} \cdot K \cdot P(0, t_i^{\text{fix}})
$$

where $K$ is the fixed rate, $L_j$ are projected LIBOR forwards, and $s$ is the floating spread.

In [ ]:
# ── QuantLib ────────────────────────────────────────────────────────────────
ql_today = QL.Date(15, 1, 2024)
QL.Settings.instance().evaluationDate = ql_today

ql_curve = QL.FlatForward(ql_today, 0.05, QL.Actual365Fixed())
ql_curve_handle = QL.YieldTermStructureHandle(ql_curve)

settle = ql_today + QL.Period(2, QL.Days)
maturity = settle + QL.Period(5, QL.Years)

fixed_sched = QL.Schedule(
    settle, maturity, QL.Period(QL.Annual),
    QL.NullCalendar(), QL.Unadjusted, QL.Unadjusted,
    QL.DateGeneration.Backward, False)

float_sched = QL.Schedule(
    settle, maturity, QL.Period(QL.Semiannual),
    QL.NullCalendar(), QL.Unadjusted, QL.Unadjusted,
    QL.DateGeneration.Backward, False)

idx = QL.Euribor6M(ql_curve_handle)

# Payer swap at 5%
ql_swap = QL.VanillaSwap(
    QL.VanillaSwap.Payer, 1_000_000.0,
    fixed_sched, 0.05, QL.Actual365Fixed(),
    float_sched, idx, 0.0, QL.Actual360())

engine = QL.DiscountingSwapEngine(ql_curve_handle)
ql_swap.setPricingEngine(engine)

print(f"QL Swap NPV      : {ql_swap.NPV():.6f}")
print(f"QL Fair rate     : {ql_swap.fairRate():.10f}")
print(f"QL Fixed leg NPV : {ql_swap.fixedLegNPV():.6f}")
print(f"QL Float leg NPV : {ql_swap.floatingLegNPV():.6f}")

In [ ]:
# ── ql-jax ──────────────────────────────────────────────────────────────────
from ql_jax.time.date import Date
from ql_jax.time.schedule import MakeSchedule
from ql_jax.time.calendar import NullCalendar
from ql_jax._util.types import BusinessDayConvention, Frequency, TimeUnit
from ql_jax.instruments.swap import make_vanilla_swap, SwapType
from ql_jax.indexes.ibor import Euribor
from ql_jax.engines.swap.discounting import (
    discounting_swap_npv, discounting_swap_fair_rate
)
from ql_jax.termstructures.yield_.flat_forward import FlatForward

jax_today = Date(15, 1, 2024)
jax_curve = FlatForward(jax_today, 0.05)
jax_settle = jax_today.advance(2, TimeUnit.Days)
jax_mat = jax_settle.advance(5, TimeUnit.Years)

fixed_sched_jax = (MakeSchedule()
    .from_date(jax_settle).to_date(jax_mat)
    .with_frequency(Frequency.Annual)
    .with_calendar(NullCalendar())
    .with_convention(BusinessDayConvention.Unadjusted)
    .build())

float_sched_jax = (MakeSchedule()
    .from_date(jax_settle).to_date(jax_mat)
    .with_frequency(Frequency.Semiannual)
    .with_calendar(NullCalendar())
    .with_convention(BusinessDayConvention.Unadjusted)
    .build())

jax_swap = make_vanilla_swap(
    SwapType.Payer, 1_000_000.0,
    fixed_sched_jax, 0.05, "Actual/365 (Fixed)",
    float_sched_jax, Euribor(6), spread=0.0)

jax_npv       = float(discounting_swap_npv(jax_swap, jax_curve))
jax_fair_rate = float(discounting_swap_fair_rate(jax_swap, jax_curve))

print(f"JAX Swap NPV      : {jax_npv:.6f}")
print(f"JAX Fair rate     : {jax_fair_rate:.10f}")

In [ ]:
compare_table([
    ("Swap NPV",   ql_swap.NPV(),      jax_npv),
    ("Fair rate",   ql_swap.fairRate(),  jax_fair_rate),
], title="5Y Payer IRS (fixed 5%, flat 5% curve)")

---
## 2. Payer vs Receiver

In [ ]:
# At-the-money swap: fixed rate = fair rate
# Off-market: fixed rate ≠ fair rate
test_rates = [0.03, 0.04, 0.05, 0.06, 0.07]

rows = []
for fixed_rate in test_rates:
    # QuantLib
    ql_s = QL.VanillaSwap(
        QL.VanillaSwap.Payer, 1_000_000.0,
        fixed_sched, fixed_rate, QL.Actual365Fixed(),
        float_sched, idx, 0.0, QL.Actual360())
    ql_s.setPricingEngine(engine)
    
    # ql-jax
    jax_s = make_vanilla_swap(
        SwapType.Payer, 1_000_000.0,
        fixed_sched_jax, fixed_rate, "Actual/365 (Fixed)",
        float_sched_jax, Euribor(6), spread=0.0)
    jax_npv_s = float(discounting_swap_npv(jax_s, jax_curve))
    
    rows.append((f"Payer {fixed_rate:.0%}", ql_s.NPV(), jax_npv_s))

compare_table(rows, title="Swap NPV at Various Fixed Rates")

---
## 3. Cashflow Inspection

In [ ]:
# Fixed leg cashflows
from ql_jax.time.daycounter import year_fraction

print("Fixed Leg Cashflows:")
print(f"{'#':>3s}  {'Payment Date':>14s}  {'Amount':>14s}")
print("-" * 40)
for i, cf in enumerate(jax_swap.fixed_leg):
    d = cf.payment_date
    date_str = f"{d.year:04d}-{d.month:02d}-{d.day:02d}"
    print(f"{i:3d}  {date_str:>14s}  {float(cf.amount):>14.2f}")

print(f"\nFloating Leg Cashflows:")
print(f"{'#':>3s}  {'Payment Date':>14s}  {'Projected Rate':>14s}  {'Amount':>14s}")
print("-" * 55)
for i, cf in enumerate(jax_swap.floating_leg):
    d = cf.payment_date
    date_str = f"{d.year:04d}-{d.month:02d}-{d.day:02d}"
    rate = float(cf.adjusted_fixing(jax_curve))
    amt = float(cf.amount_with_curve(jax_curve))
    print(f"{i:3d}  {date_str:>14s}  {rate:>14.6f}  {amt:>14.2f}")

---
## 4. DV01 via AD

**DV01** (dollar value of a basis point) is the change in NPV for a 1bp parallel shift:

$$
\text{DV01} = \frac{\partial \text{NPV}}{\partial r} \times 0.0001
$$

With JAX AD, this is exact — no finite-difference approximation.

In [ ]:
def swap_npv_from_rate(r):
    curve = FlatForward(jax_today, r)
    return discounting_swap_npv(jax_swap, curve)

r0 = jnp.float64(0.05)
npv_base = float(swap_npv_from_rate(r0))

# AD DV01
dNPV_dr = float(jax.grad(swap_npv_from_rate)(r0))
dv01_ad = dNPV_dr * 0.0001

# FD DV01 for comparison
npv_up = float(swap_npv_from_rate(r0 + 0.0001))
dv01_fd = npv_up - npv_base

print(f"Swap NPV        : {npv_base:.6f}")
print(f"dNPV/dr (AD)    : {dNPV_dr:.2f}")
print(f"DV01 (AD)       : {dv01_ad:.2f}")
print(f"DV01 (FD 1bp)   : {dv01_fd:.2f}")
print(f"Difference      : {abs(dv01_ad - dv01_fd):.6f}")

---
## 5. Batch Pricing: 1,000 Rate Scenarios

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rate_scenarios = jnp.linspace(0.01, 0.10, 1000)

# vmap for batch
batch_npv = jax.vmap(swap_npv_from_rate)(rate_scenarios)
batch_dv01 = jax.vmap(jax.grad(swap_npv_from_rate))(rate_scenarios) * 0.0001

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(np.array(rate_scenarios) * 100, np.array(batch_npv), 'b-')
ax1.axhline(0, color='k', linewidth=0.5)
ax1.set_xlabel('Flat Rate (%)')
ax1.set_ylabel('Swap NPV')
ax1.set_title('1,000 Swap NPVs via vmap')
ax1.grid(True, alpha=0.3)

ax2.plot(np.array(rate_scenarios) * 100, np.array(batch_dv01), 'r-')
ax2.set_xlabel('Flat Rate (%)')
ax2.set_ylabel('DV01')
ax2.set_title('DV01 Across Scenarios (via jax.grad + vmap)')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---
## 6. Performance

In [ ]:
def ql_swap_npv():
    return ql_swap.NPV()

def jax_swap_npv_fn():
    return float(discounting_swap_npv(jax_swap, jax_curve))

ql_ms, _  = timed_ms(ql_swap_npv, warmup=5, runs=20)
jax_ms, _ = timed_ms(jax_swap_npv_fn, warmup=5, runs=20)

print(f"Swap NPV:")
print(f"  QuantLib : {ql_ms:.4f} ms")
print(f"  ql-jax   : {jax_ms:.4f} ms")

plot_speedup(["NPV"], [ql_ms], [jax_ms], title="Swap NPV: QuantLib vs ql-jax")

---
## 7. Exercises

1. **Par rate sensitivity**: Compute $\partial K_{\text{fair}} / \partial r_i$ for each bootstrap pillar rate using `jax.jacrev`.
2. **Gamma**: Compute $\partial^2 \text{NPV} / \partial r^2$ via `jax.grad(jax.grad(...))`. How does swap gamma compare to bond convexity?
3. **Multi-curve**: Build separate OIS (discount) and EURIBOR (forecast) curves and price a swap using both.

---
*End of Notebook 04*